# Enriched Post-Double Selection (EPDS)
### Simulation Study — Main Notebook

**Structure:**
1. Setup and imports
2. Single DGP exploration (sanity checks)
3. Single estimator walkthrough
4. Monte Carlo simulation — all DGPs × all estimators
5. Results and figures
6. PySR functional form recovery

---

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.stats import norm
from IPython.display import display
import pandas as pd
import os
from estimators import epds

import sys
sys.path.append('.')

from dgp        import dgp1, dgp2, dgp3, dgp4, dgp5, DGP_REGISTRY
from estimators import (naive_ols, full_ols, pds_lasso,
                         dml_lasso, dml_nn, ESTIMATOR_REGISTRY)
from simulation import run_simulation, run_all
from evaluate   import evaluate, evaluate_all, summary_table, print_summary
from plots      import (plot_distributions, plot_coverage,
                         plot_bias, plot_rmse_heatmap, plot_summary_panel)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# global simulation parameters
N      = 500
P      = 50
S      = 6
BETA0  = 0.5
N_REPS = 500   # set lower (e.g. 50) for quick runs during development, 500 when productionised

print('Setup complete')

Setup complete


---
## 2. Single DGP exploration
Inspect each DGP before running simulations — sanity check shapes, distributions, and confounding structure.

In [ ]:
# inspect all DGPs
for name, entry in DGP_REGISTRY.items():
    X, d, y, beta0 = entry['fn'](n=N, p=P, s=S, beta0=BETA0, seed=42)
    print(f"{name}  ({entry['label']})")
    print(f"  y: mean={y.mean():.3f}  std={y.std():.3f}")
    print(f"  d: mean={d.mean():.3f}  std={d.std():.3f}")
    print(f"  corr(d,y): {np.corrcoef(d,y)[0,1]:.3f}")
    print()

In [ ]:
# visualise DGP1 -- scatter x1 vs y
X, d, y, _ = dgp1(n=N, p=P, s=S, beta0=BETA0, seed=42)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

for ax, j, lbl in zip(axes, [0, 1, 5], ['x₁ (γ=1.0)', 'x₂ (γ=-0.8)', 'x₆ (γ=0)']):
    ax.scatter(X[:, j], y, alpha=0.2, s=10, color='#534AB7')
    m, b = np.polyfit(X[:, j], y, 1)
    xr = np.linspace(X[:, j].min(), X[:, j].max(), 100)
    ax.plot(xr, m * xr + b, color='#D85A30', linewidth=1.5)
    ax.set_xlabel(lbl)
    ax.set_ylabel('y')
    ax.set_title(f'slope ≈ {m:.2f}')

fig.suptitle('DGP1 — x vs y relationships', fontweight='500')
plt.tight_layout()

In [ ]:
# visualise DGP2 -- nonlinear relationships
X, d, y, _ = dgp2(n=N, p=P, s=S, beta0=BETA0, seed=42)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

for ax, j, lbl in zip(axes, [0, 1, 2], ['x₁ (γ=x₁²)', 'x₂ (γ=0.8, linear)', 'x₃ (γ=x₃²)']):
    ax.scatter(X[:, j], y, alpha=0.2, s=10, color='#1D9E75')
    ax.set_xlabel(lbl)
    ax.set_ylabel('y')

fig.suptitle('DGP2 — quadratic relationships', fontweight='500')
plt.tight_layout()

---
## 3. Single estimator walkthrough
Step through PDS-LASSO manually to understand what gets selected.

In [ ]:
# DGP1 -- manual PDS-LASSO walkthrough
X, d, y, beta0 = dgp1(n=N, p=P, s=S, beta0=BETA0, seed=42)

def theoretical_lambda(n, p, c=1.1, alpha=0.05):
    return (c / np.sqrt(n)) * norm.ppf(1 - alpha / (2 * p))

lam = theoretical_lambda(N, P)
print(f'Theoretical lambda: {lam:.4f}')

# LASSO y on X
lasso_y = Lasso(alpha=lam, max_iter=10000).fit(X, y)
S_y     = set(np.where(lasso_y.coef_ != 0)[0])

# LASSO d on X
lasso_d = Lasso(alpha=lam, max_iter=10000).fit(X, d)
S_d     = set(np.where(lasso_d.coef_ != 0)[0])

S_union = sorted(S_y | S_d)

print(f'S_y    (from y ~ X): {sorted(S_y)}')
print(f'S_d    (from d ~ X): {sorted(S_d)}')
print(f'S_union            : {S_union}')
print(f'True nonzero       : {list(range(S))}')

In [ ]:
# post-selection OLS
col_names = ['d'] + [f'x{i+1}' for i in range(P)]
df_data   = pd.DataFrame(np.column_stack([d, X]), columns=col_names)
df_data['y'] = y

X_pds   = sm.add_constant(df_data[['d'] + [f'x{i+1}' for i in S_union]])
pds_res = sm.OLS(df_data['y'], X_pds).fit(cov_type='HC1')
print(pds_res.summary())
print(f'\nTrue beta0 = {beta0}')

---
## 4. Monte Carlo simulation
Run all DGP × estimator combinations.

> **Note:** Set `N_REPS = 50` for a quick development run. Use `N_REPS = 500` for final results.

In [ ]:
# define which DGPs and estimators to run
# exclude epds here until PySR pipeline is validated
DGP_KEYS = ['dgp1', 'dgp2', 'dgp3', 'dgp4', 'dgp5']
EST_KEYS  = ['naive_ols', 'pds_lasso', 'dml_lasso', 'dml_nn']

combined = run_all(
    dgp_registry       = DGP_REGISTRY,
    estimator_registry = ESTIMATOR_REGISTRY,
    dgp_keys           = DGP_KEYS,
    estimator_keys     = EST_KEYS,
    n                  = N,
    p                  = P,
    s                  = S,
    beta0              = BETA0,
    n_reps             = N_REPS,
    n_jobs             = 1,     # set to -1 to use all cores
    save_dir           = '../results',
)

print(f'\nTotal rows: {len(combined)}')
print(combined.groupby(['dgp', 'estimator'])['beta_hat'].count())

---
## 5. Results

In [ ]:
# compute metrics
metrics = evaluate_all(combined, beta0=BETA0)
print_summary(metrics, beta0=BETA0)

In [ ]:
# RMSE table -- publication ready
print('RMSE')
print(summary_table(metrics, metric='rmse').to_string())
print()
print('Coverage')
print(summary_table(metrics, metric='coverage').to_string())

In [ ]:
# distribution plots for each DGP
for dgp_key in DGP_KEYS:
    fig = plot_distributions(combined, beta0=BETA0, dgp_key=dgp_key)
    plt.show()

In [ ]:
# coverage bar chart
fig = plot_coverage(metrics)
plt.show()

In [ ]:
# bias plot
fig = plot_bias(metrics)
plt.show()

In [ ]:
# RMSE heatmap
fig = plot_rmse_heatmap(metrics)
plt.show()

In [ ]:
# 2x2 summary panel
fig = plot_summary_panel(metrics)
plt.show()

---
## 6. PySR functional form recovery
Run PySR on each DGP to check what functional forms it recovers.
This is the enrichment step of EPDS.

In [ ]:
from pysr import PySRRegressor

def run_pysr(dgp_fn, dgp_label, n=500, p=50, s=6, beta0=0.5, seed=42,
             niterations=40):
    """Run PySR on a DGP and print recovered equations."""
    print(f'\n{"-"*60}')
    print(f'DGP: {dgp_label}')
    print(f'{"-"*60}')

    X, d, y, _ = dgp_fn(n=n, p=p, s=s, beta0=beta0, seed=seed)

    # use selected variables only -- in practice comes from NFSRD
    # here we cheat and use true support for illustration
    X_sel  = X[:, :s]
    X_pysr = np.column_stack([d, X_sel])
    cols   = ['d'] + [f'x{i+1}' for i in range(s)]

    model = PySRRegressor(
        niterations    = niterations,
        binary_operators = ['+', '-', '*'],
        unary_operators  = ['square', 'log', 'sqrt'],
        maxsize          = 30,
        parsimony        = 0.0005,
        procs            = 0,
        random_state     = 42,
        verbosity        = 0,
    )
    model.fit(X_pysr, y, variable_names=cols)

    print(f'Best equation : {model.sympy()}')
    print(f'LaTeX         : {model.latex()}')
    print(f'\nPareto frontier:')
    print(model.equations_[['complexity', 'loss', 'score', 'equation']]
          .to_string(index=False))

    return model

In [ ]:
# DGP1 -- should recover linear equation
model_dgp1 = run_pysr(dgp1, 'Linear sparse')

In [ ]:
# DGP2 -- should recover x1^2 and x3^2
model_dgp2 = run_pysr(dgp2, 'Quadratic + linear', s=6)

In [ ]:
# DGP3 -- should recover x1*x2 and x4*x5
model_dgp3 = run_pysr(dgp3, 'Interactions + linear', s=6)

In [ ]:
# DGP4 -- should recover x1^2*x2, log(x3), x4*x5
model_dgp4 = run_pysr(dgp4, 'Mixed nonlinear', s=6, niterations=80)

---
## 7. EPDS full run
Once PySR recovery looks good above, run EPDS in the Monte Carlo simulation.

In [2]:
# ============================================================
# LOAD CACHED NON-EPDS RESULTS
# ============================================================
cache_path = '../results/combined_non_epds.csv'

if os.path.exists(cache_path):
    combined = pd.read_csv(cache_path)
    print(f"Loaded {len(combined)} cached rows")
    print(combined.groupby(['dgp', 'estimator'])['beta_hat'].count())
else:
    print("No cache -- run simulation cells first")

Loaded 10000 cached rows
dgp   estimator
dgp1  dml_lasso    500
      dml_nn       500
      naive_ols    500
      pds_lasso    500
dgp2  dml_lasso    500
      dml_nn       500
      naive_ols    500
      pds_lasso    500
dgp3  dml_lasso    500
      dml_nn       500
      naive_ols    500
      pds_lasso    500
dgp4  dml_lasso    500
      dml_nn       500
      naive_ols    500
      pds_lasso    500
dgp5  dml_lasso    500
      dml_nn       500
      naive_ols    500
      pds_lasso    500
Name: beta_hat, dtype: int64


In [ ]:
from estimators import epds, EPDS_LOG

# clear log before run
EPDS_LOG.clear()

# run with log=True
epds_log = lambda X, d, y, n, p: epds(X, d, y, n, p, log=True)

epds_results = run_all(
    dgp_registry       = DGP_REGISTRY,
    estimator_registry = {'epds': {'fn': epds_log, 'label': 'EPDS'}},
    dgp_keys           = ['dgp1','dgp2','dgp3','dgp4','dgp5'],
    n=N, p=P, s=S, beta0=BETA0,
    n_reps             = 1,
    save_dir           = '../results',
)

# inspect after run
import pandas as pd
log_df = pd.DataFrame(EPDS_LOG)
print(log_df.groupby(level=0).agg({'terms_added': 'mean', 
                                    'n_selected': 'mean',
                                    'loss_y': 'mean'}))
log_df.to_csv('../results/epds_log.csv', index=False)


Running: Linear sparse × EPDS


Replications:   0%|          | 0/500 [00:00<?, ?it/s]

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


/Users/przemekszkodon/miniconda3/envs/epds/lib/python3.11/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
/Users/przemekszkodon/miniconda3/envs/epds/lib/python3.11/site-packages/pysr/sr.py:1873: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(
/Users/przemekszkodon/miniconda3/envs/epds/lib/python3.11/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
/Users/przemekszkodon/miniconda3/envs/epds/lib/python3.11/site-packages/pysr/sr.py:1873: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(
Replications:   0%|          | 1/500 [00:12<1:47:37, 12.94s/it]/Users/pr

  Saved to ../results/dgp1_epds.csv

Running: Quadratic + linear × EPDS


...

Replications:   0%|          | 0/500 [00:00<?, ?it/s]/Users/przemekszkodon/miniconda3/envs/epds/lib/python3.11/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
/Users/przemekszkodon/miniconda3/envs/epds/lib/python3.11/site-packages/pysr/sr.py:1873: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(
/Users/przemekszkodon/miniconda3/envs/epds/lib/python3.11/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
/Users/przemekszkodon/miniconda3/envs/epds/lib/python3.11/site-packages/pysr/sr.py:1873: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(
Replications:   0%|

  Saved to ../results/dgp4_epds.csv

Running: Mixed nonlinear + confounding × EPDS


Replications:   0%|          | 0/500 [00:00<?, ?it/s]/Users/przemekszkodon/miniconda3/envs/epds/lib/python3.11/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
/Users/przemekszkodon/miniconda3/envs/epds/lib/python3.11/site-packages/pysr/sr.py:1873: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(
/Users/przemekszkodon/miniconda3/envs/epds/lib/python3.11/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
/Users/przemekszkodon/miniconda3/envs/epds/lib/python3.11/site-packages/pysr/sr.py:1873: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(
Replications:   0%|

  Saved to ../results/dgp5_epds.csv

All results saved to ../results/all_results.csv


KeyError: "Label(s) ['loss_y', 'n_selected', 'terms_added'] do not exist"

In [6]:
# load EPDS results
epds_results = pd.read_csv('../results/all_results.csv')
epds_results = epds_results[epds_results['estimator'] == 'epds']
epds_results['est_label'] = 'EPDS'

# combine with cached non-EPDS
all_results = pd.concat([combined, epds_results], ignore_index=True)
all_metrics = evaluate_all(all_results, beta0=BETA0)
print_summary(all_metrics, beta0=BETA0)


True ATE: 0.5

BIAS
------------------------------------------------------------
                               DML-LASSO  DML-NN  EPDS  Naive OLS  PDS-LASSO
DGP                                                                         
Interactions + linear            -0.0057  0.0967   NaN     0.0146    -0.0276
Linear sparse                     0.0013  0.0885   NaN    -0.0055     0.0001
Mixed nonlinear                   0.0264  0.1653   NaN     0.0222     0.0128
Mixed nonlinear + confounding     0.0200  0.2049   NaN     0.3037     0.0140
Quadratic + linear                0.0004  0.0378   NaN     0.0032    -0.0158

RMSE
------------------------------------------------------------
                               DML-LASSO  DML-NN  EPDS  Naive OLS  PDS-LASSO
DGP                                                                         
Interactions + linear             0.2652  0.2935   NaN     0.3560     0.2583
Linear sparse                     0.0489  0.1424   NaN     0.3640     0.0454
Mixe

In [10]:
import glob

epds_files = glob.glob('../results/dgp*_epds.csv')
print(epds_files)

dfs = []
for f in epds_files:
    df = pd.read_csv(f)
    print(f"{f}: {len(df)} rows, {df['beta_hat'].isna().sum()} NaN")
    dfs.append(df)

epds_results = pd.concat(dfs, ignore_index=True)
print(f"\nTotal: {len(epds_results)} rows")
print(f"Failed: {epds_results['failed'].sum()}")
print(f"NaN beta_hat: {epds_results['beta_hat'].isna().sum()}")

['../results/dgp1_epds.csv', '../results/dgp2_epds.csv', '../results/dgp3_epds.csv', '../results/dgp5_epds.csv', '../results/dgp4_epds.csv']
../results/dgp1_epds.csv: 500 rows, 500 NaN
../results/dgp2_epds.csv: 500 rows, 500 NaN
../results/dgp3_epds.csv: 500 rows, 500 NaN
../results/dgp5_epds.csv: 500 rows, 500 NaN
../results/dgp4_epds.csv: 500 rows, 500 NaN

Total: 2500 rows
Failed: 2500
NaN beta_hat: 2500


In [7]:
# ============================================================
# COMPREHENSIVE COMPARISON TABLE
# ============================================================

equations = {
    'Linear sparse'                : 'y = 0.5d + 5x1 - 5x2 + 3x3 - 3x4 + x5 - x6 + e',
    'Quadratic + linear'           : 'y = 0.5d + 5x1^2 - 5x2 + 3x3^2 - 3x4 + x5 + e',
    'Interactions + linear'        : 'y = 0.5d + 5x1*x2 - 5x3 + 3x4*x5 - 3x6 + e',
    'Mixed nonlinear'              : 'y = 0.5d + 5x1^2*x2 - 5log|x3| + 3x4*x5 - 3x6 + e',
    'Mixed nonlinear + confounding': 'y = 0.5d + 5x1^2*x2 - 5log|x3| + 3x4*x5 - 3x6 + e, d = Xd + v',
}

# ordered columns -- PDS-LASSO and EPDS adjacent
COL_ORDER = ['Naive OLS', 'DML-LASSO', 'DML-NN', 'PDS-LASSO', 'EPDS']

print(f'True ATE = {BETA0}  |  n = {N}  |  p = {P}  |  s = {S}  |  reps = {N_REPS}\n')

for metric, caption, cmap in [
    ('bias',     'Bias',                        'RdYlGn_r'),
    ('rmse',     'RMSE',                        'RdYlGn_r'),
    ('coverage', 'Coverage (nominal = 0.95)',   'RdYlGn'),
    ('size',     'Size',                        'RdYlGn_r'),
]:
    pivot = summary_table(all_metrics, metric=metric)

    # reorder columns
    available = [c for c in COL_ORDER if c in pivot.columns]
    pivot     = pivot[available]

    # add equation column
    pivot.insert(0, 'True equation', [equations.get(i, '') for i in pivot.index])

    numeric_cols = [c for c in pivot.columns if c != 'True equation']

    display(
        pivot.style
        .set_caption(caption)
        .format({c: '{:.4f}' for c in numeric_cols})
        .background_gradient(subset=numeric_cols, cmap=cmap, axis=None)
    )

True ATE = 0.5  |  n = 500  |  p = 50  |  s = 6  |  reps = 500



,True equation,Naive OLS,DML-LASSO,DML-NN,PDS-LASSO,EPDS
DGP,,,,,,
Interactions + linear,y = 0.5d + 5x1*x2 - 5x3 + 3x4*x5 - 3x6 + e,0.0146,-0.0057,0.0967,-0.0276,nan
Linear sparse,y = 0.5d + 5x1 - 5x2 + 3x3 - 3x4 + x5 - x6 + e,-0.0055,0.0013,0.0885,0.0001,nan
Mixed nonlinear,y = 0.5d + 5x1^2*x2 - 5log|x3| + 3x4*x5 - 3x6 + e,0.0222,0.0264,0.1653,0.0128,nan
Mixed nonlinear + confounding,"y = 0.5d + 5x1^2*x2 - 5log|x3| + 3x4*x5 - 3x6 + e, d = Xd + v",0.3037,0.0200,0.2049,0.0140,nan
Quadratic + linear,y = 0.5d + 5x1^2 - 5x2 + 3x3^2 - 3x4 + x5 + e,0.0032,0.0004,0.0378,-0.0158,nan


,True equation,Naive OLS,DML-LASSO,DML-NN,PDS-LASSO,EPDS
DGP,,,,,,
Interactions + linear,y = 0.5d + 5x1*x2 - 5x3 + 3x4*x5 - 3x6 + e,0.3560,0.2652,0.2935,0.2583,nan
Linear sparse,y = 0.5d + 5x1 - 5x2 + 3x3 - 3x4 + x5 - x6 + e,0.3640,0.0489,0.1424,0.0454,nan
Mixed nonlinear,y = 0.5d + 5x1^2*x2 - 5log|x3| + 3x4*x5 - 3x6 + e,0.4688,0.4094,0.4598,0.4032,nan
Mixed nonlinear + confounding,"y = 0.5d + 5x1^2*x2 - 5log|x3| + 3x4*x5 - 3x6 + e, d = Xd + v",0.3839,0.3597,0.3988,0.4042,nan
Quadratic + linear,y = 0.5d + 5x1^2 - 5x2 + 3x3^2 - 3x4 + x5 + e,0.4588,0.3829,0.3901,0.3766,nan


,True equation,Naive OLS,DML-LASSO,DML-NN,PDS-LASSO,EPDS
DGP,,,,,,
Interactions + linear,y = 0.5d + 5x1*x2 - 5x3 + 3x4*x5 - 3x6 + e,0.9540,0.9680,0.9300,0.9640,nan
Linear sparse,y = 0.5d + 5x1 - 5x2 + 3x3 - 3x4 + x5 - x6 + e,0.9500,0.9560,0.8940,0.9520,nan
Mixed nonlinear,y = 0.5d + 5x1^2*x2 - 5log|x3| + 3x4*x5 - 3x6 + e,0.9500,0.9580,0.9480,0.9580,nan
Mixed nonlinear + confounding,"y = 0.5d + 5x1^2*x2 - 5log|x3| + 3x4*x5 - 3x6 + e, d = Xd + v",0.7480,0.9680,0.9180,0.9580,nan
Quadratic + linear,y = 0.5d + 5x1^2 - 5x2 + 3x3^2 - 3x4 + x5 + e,0.9540,0.9420,0.9460,0.9440,nan


,True equation,Naive OLS,DML-LASSO,DML-NN,PDS-LASSO,EPDS
DGP,,,,,,
Interactions + linear,y = 0.5d + 5x1*x2 - 5x3 + 3x4*x5 - 3x6 + e,0.2940,0.4720,0.6000,0.4500,nan
Linear sparse,y = 0.5d + 5x1 - 5x2 + 3x3 - 3x4 + x5 - x6 + e,0.2520,1.0000,0.9920,1.0000,nan
Mixed nonlinear,y = 0.5d + 5x1^2*x2 - 5log|x3| + 3x4*x5 - 3x6 + e,0.1700,0.1920,0.3280,0.1940,nan
Mixed nonlinear + confounding,"y = 0.5d + 5x1^2*x2 - 5log|x3| + 3x4*x5 - 3x6 + e, d = Xd + v",0.9600,0.1880,0.5460,0.1880,nan
Quadratic + linear,y = 0.5d + 5x1^2 - 5x2 + 3x3^2 - 3x4 + x5 + e,0.2100,0.2620,0.3180,0.2600,nan


In [ ]:
# ============================================================
# CACHE NON-EPDS RESULTS -- run this once after your 500 reps
# ============================================================

"""
os.makedirs('../results', exist_ok=True)

combined.to_csv('../results/combined_non_epds.csv', index=False)
print(f"Cached {len(combined)} rows")
print(combined.groupby(['dgp', 'estimator'])['beta_hat'].count())
"""